# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
# 🛠️ TOOL 3 (BONUS): Word Counter

def count_words(text: str) -> int:
    """Count the number of words in a piece of text."""
    try:
        return len(text.split())
    except Exception:
        return 0

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [4]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re
import logging

# Configure basic logging so we can see the agent's routing decisions
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")


def agent(query: str):
    """
    Single-Agent Smart Assistant.

    Routes the incoming query to the correct tool based on intent:
      - "calculate"    -> Calculator Tool
      - "keywords"     -> Keyword Extractor Tool
      - "count words"  -> Word Counter Tool (bonus)
      - anything else  -> General/direct response

    Always returns a structured dict:
        { "type": "calculation" | "keywords" | "word_count" | "general" | "error", "result": ... }
    """
    try:
        if not isinstance(query, str) or not query.strip():
            logger.warning("Empty or invalid query received.")
            return {"type": "error", "result": "Query must be a non-empty string."}

        query_lower = query.lower()

        # ---------- Route 1: Calculator ----------
        if "calculate" in query_lower:
            logger.info("Routing to Calculator Tool.")

            # Pull out just the math-looking part of the query
            # (digits, decimal points, parentheses and + - * / operators)
            match = re.findall(r"[\d\.\+\-\*/\(\)\s]+", query)
            expression = "".join(match).strip()

            if not expression:
                logger.warning("No valid mathematical expression found in query.")
                return {"type": "error", "result": "No valid expression found to calculate."}

            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}

            return {"type": "calculation", "result": calc_result}

        # ---------- Route 2: Keyword Extractor ----------
        elif "keywords" in query_lower:
            logger.info("Routing to Keyword Extractor Tool.")

            # Use the text after "from" if present, else use the whole query
            if "from" in query_lower:
                idx = query_lower.index("from") + len("from")
                text = query[idx:].strip()
            else:
                text = query.strip()

            if not text:
                logger.warning("No text found to extract keywords from.")
                return {"type": "error", "result": "No text found to extract keywords from."}

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # ---------- Route 3 (BONUS): Word Counter ----------
        elif "count words" in query_lower or "word count" in query_lower:
            logger.info("Routing to Word Counter Tool.")

            for phrase in ["count words in", "count words", "word count of", "word count"]:
                if phrase in query_lower:
                    idx = query_lower.index(phrase) + len(phrase)
                    text = query[idx:].strip()
                    break
            else:
                text = query.strip()

            if not text:
                logger.warning("No text found to count words in.")
                return {"type": "error", "result": "No text found to count words in."}

            return {"type": "word_count", "result": count_words(text)}

        # ---------- Route 4: General response ----------
        else:
            logger.info("Routing to General response.")
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. I can help with calculations "
                           f"(e.g. 'Calculate 5 + 3'), keyword extraction "
                           f"(e.g. 'Extract keywords from ...'), or word counts "
                           f"(e.g. 'Count words in ...')."
            }

    except Exception as e:
        logger.error(f"Unexpected error while processing query: {e}")
        return {"type": "error", "result": f"Unexpected error: {e}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [5]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Count words in The quick brown fox jumps over the lazy dog",  # bonus tool
    "Calculate 10 / 0",   # error handling test
    "",                   # empty query test
]

for q in queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

2026-07-12 19:00:09,280 | INFO | Routing to Calculator Tool.


Query: 'Calculate 20 + 5'
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: 'Extract keywords from Artificial Intelligence is transforming industries'


2026-07-12 19:00:09,282 | INFO | Routing to Keyword Extractor Tool.
2026-07-12 19:00:09,285 | INFO | Routing to General response.
2026-07-12 19:00:09,287 | INFO | Routing to Word Counter Tool.
2026-07-12 19:00:09,290 | INFO | Routing to Calculator Tool.
2026-07-12 19:00:09,290 | WARNING | Empty or invalid query received.


Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'industries', 'transforming']}
--------------------------------------------------
Query: 'What is machine learning?'
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. I can help with calculations (e.g. 'Calculate 5 + 3'), keyword extraction (e.g. 'Extract keywords from ...'), or word counts (e.g. 'Count words in ...')."}
--------------------------------------------------
Query: 'Count words in The quick brown fox jumps over the lazy dog'
Response: {'type': 'word_count', 'result': 9}
--------------------------------------------------
Query: 'Calculate 10 / 0'
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: ''
Response: {'type': 'error', 'result': 'Query must be a non-empty string.'}
--------------------------------------------------


In [7]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

2026-07-12 19:07:15,683 | INFO | Routing to Calculator Tool.


Response: {'type': 'calculation', 'result': '72'}


2026-07-12 19:07:26,879 | INFO | Routing to General response.


Response: {'type': 'general', 'result': "I received your query: 'exit '. I can help with calculations (e.g. 'Calculate 5 + 3'), keyword extraction (e.g. 'Extract keywords from ...'), or word counts (e.g. 'Count words in ...')."}


VINIT GAUTAM 
PIET 
WEEK 8 ASSIGNMENT 
